# Homework 2: Data Processing

This notebook focuses on data processing and model training techniques including:
- Loading preprocessed data
- Feature engineering and selection
- Model training and evaluation
- Cross-validation
- Hyperparameter tuning

## 1. Setup and Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Custom database module
import sys
sys.path.append('..')
from database.data_loader import DataLoader

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 2. Load Preprocessed Data

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load preprocessed data from Homework 1
# df = loader.load_csv('preprocessed_data.csv')

# For demonstration, create sample data
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                           n_redundant=3, random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Feature Engineering

In [ ]:
# Create new features (example: interactions, polynomials)
df['feature_interaction'] = df['feature_0'] * df['feature_1']
df['feature_squared'] = df['feature_0'] ** 2

print("New features created:")
print(df[['feature_0', 'feature_1', 'feature_interaction', 'feature_squared']].head())

## 4. Feature Selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Prepare features and target
X = df.drop('target', axis=1)
y = df['target']

# Select top k features
selector = SelectKBest(score_func=f_classif, k=8)
X_selected = selector.fit_transform(X, y)

# Get selected feature names
selected_features = X.columns[selector.get_support()].tolist()
print(f"Selected features: {selected_features}")

## 5. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Class distribution in training: {np.bincount(y_train)}")

## 6. Model Training

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

# Train and evaluate each model
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

## 7. Cross-Validation

In [ ]:
# Perform cross-validation
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name} CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 8. Hyperparameter Tuning

In [ ]:
# Grid search for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

## 9. Final Model Evaluation

In [ ]:
# Use best model from grid search
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 10. Save Model and Results

In [ ]:
# Save model for later use
import joblib
# joblib.dump(best_model, 'best_model.pkl')
# Save predictions for analysis
# results_df = pd.DataFrame({'true': y_test, 'predicted': y_pred})
# loader.save_csv(results_df, 'predictions.csv')
print("Processing complete!")